# itemadapter smoke test

Tests `itemadapter` — a common interface for data container classes (dict, dataclass, attrs, pydantic, scrapy.Item).

In [ ]:
# Example 1: ItemAdapter wraps a plain dict and exposes a uniform interface.
from itemadapter import ItemAdapter

raw = {"name": "widget", "price": 9.99, "in_stock": True}
adapter = ItemAdapter(raw)

print(f"adapter['name'] = {adapter['name']}")
print(f"len(adapter)   = {len(adapter)}")
print(f"keys()         = {list(adapter.keys())}")
print(f"asdict()       = {adapter.asdict()}")

# Mutation flows back to the underlying dict.
adapter['price'] = 11.49
assert raw['price'] == 11.49
assert dict(adapter) == raw
print("dict wrap + mutation: OK")

In [ ]:
# Example 2: ItemAdapter over a dataclass — same uniform interface, field metadata preserved.
from dataclasses import dataclass, field
from itemadapter import ItemAdapter

@dataclass
class Product:
    name: str = field(default="", metadata={"serializer": str})
    price: float = field(default=0.0, metadata={"serializer": float})

p = Product(name="gizmo", price=2.50)
adapter = ItemAdapter(p)

print(f"adapter['name'] = {adapter['name']}")
print(f"field_names() = {adapter.field_names()}")

# get_field_meta exposes the dataclass field metadata mapping.
meta = adapter.get_field_meta('price')
print(f"price field meta = {dict(meta)}")

adapter['name'] = 'gadget'
assert p.name == 'gadget'
print("dataclass wrap + meta: OK")

In [ ]:
# Example 3: is_item / type detection helpers + ItemAdapter.is_item.
from dataclasses import dataclass
from itemadapter import ItemAdapter, is_item

@dataclass
class Row:
    a: int = 0

# Recognized item kinds.
assert is_item({"x": 1}) is True
assert is_item(Row(a=5)) is True
assert ItemAdapter.is_item({"x": 1}) is True
assert ItemAdapter.is_item(Row()) is True

# Non-items.
assert is_item([1, 2, 3]) is False
assert is_item("hello") is False
assert is_item(42) is False

print("is_item / ItemAdapter.is_item: OK")

In [ ]:
# Example 4: dict-like methods (get, pop, update, contains, iteration).
from itemadapter import ItemAdapter

adapter = ItemAdapter({"a": 1, "b": 2, "c": 3})

assert adapter.get('a') == 1
assert adapter.get('missing', 'fallback') == 'fallback'
assert 'b' in adapter
assert 'missing' not in adapter

adapter.update({"c": 30, "d": 4})
assert adapter['c'] == 30
assert adapter['d'] == 4

popped = adapter.pop('a')
assert popped == 1
assert 'a' not in adapter

items_seen = sorted(adapter.items())
print(f"items = {items_seen}")
print("dict-like methods: OK")

In [ ]:
# Example 5: ItemAdapter.get_field_meta_from_class for static-class introspection,
# plus KeyError on unknown fields for dataclass-backed adapters.
from dataclasses import dataclass, field
from itemadapter import ItemAdapter

@dataclass
class Book:
    title: str = field(default="", metadata={"lang": "en"})
    pages: int = field(default=0)

# Class-level metadata access (no instance required).
title_meta = ItemAdapter.get_field_meta_from_class(Book, 'title')
print(f"Book.title meta = {dict(title_meta)}")
assert title_meta.get('lang') == 'en'

# Unknown field on a dataclass-backed adapter raises KeyError on assignment.
adapter = ItemAdapter(Book(title='Dune', pages=412))
raised = False
try:
    adapter['nonexistent_field'] = 'x'
except KeyError:
    raised = True
assert raised, "expected KeyError on unknown dataclass field"

print("class-level meta + KeyError guard: OK")